In [1]:
megadescriptor_version = 'T-224'  # 'S-224', 'B-224', 'L-384'
detection = '' # _detected', '_detected_manual'

In [2]:
import torch

# Load the saved embeddings
embeddings = f'saved_models/{megadescriptor_version}/embeddings/emb{detection}.pt'
labels = f'saved_models/{megadescriptor_version}/labels/labels{detection}.pt'
label_encoder = f'saved_models/{megadescriptor_version}/label_encoders/label_encoder{detection}.pkl'

all_embeddings = torch.load(embeddings)

print(all_embeddings.shape)  # torch.Size([260, 768])


torch.Size([319, 768])


In [3]:
import torch, joblib
label_ids = torch.load(labels, weights_only=False)
encoder = joblib.load(label_encoder)

# Convert names back later:
names = encoder.inverse_transform(label_ids)


In [4]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# Example setup
X = all_embeddings.float()
y = torch.from_numpy(label_ids).long()   # shape (260,)

dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)


In [5]:
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=10, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [6]:
model = Classifier(input_dim=768, num_classes=len(torch.unique(y)))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(100):
    if epoch != 0:
        model.train()
    total_loss = 0
    correct = 0
    for X_batch, y_batch in train_loader:
        if epoch != 0:
            optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        if epoch != 0:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        correct += (outputs.argmax(1) == y_batch).sum().item()

    acc = correct / len(dataset)
    print(f"Epoch {epoch}: loss={total_loss:.3f}, acc={acc:.3f}")


Epoch 0: loss=52.697, acc=0.091
Epoch 1: loss=49.768, acc=0.204
Epoch 2: loss=44.088, acc=0.241
Epoch 3: loss=39.287, acc=0.382
Epoch 4: loss=35.258, acc=0.455
Epoch 5: loss=29.749, acc=0.520
Epoch 6: loss=24.089, acc=0.602
Epoch 7: loss=22.014, acc=0.652
Epoch 8: loss=18.168, acc=0.699
Epoch 9: loss=15.282, acc=0.752
Epoch 10: loss=13.226, acc=0.768
Epoch 11: loss=11.692, acc=0.812
Epoch 12: loss=10.291, acc=0.843
Epoch 13: loss=9.296, acc=0.853
Epoch 14: loss=8.434, acc=0.865
Epoch 15: loss=8.097, acc=0.862
Epoch 16: loss=6.804, acc=0.890
Epoch 17: loss=6.688, acc=0.881
Epoch 18: loss=5.926, acc=0.915
Epoch 19: loss=5.042, acc=0.915
Epoch 20: loss=5.987, acc=0.912
Epoch 21: loss=4.791, acc=0.912
Epoch 22: loss=4.296, acc=0.944
Epoch 23: loss=4.216, acc=0.934
Epoch 24: loss=4.560, acc=0.931
Epoch 25: loss=3.687, acc=0.931
Epoch 26: loss=3.566, acc=0.931
Epoch 27: loss=2.821, acc=0.962
Epoch 28: loss=4.362, acc=0.956
Epoch 29: loss=2.231, acc=0.969
Epoch 30: loss=2.651, acc=0.953
Epoch

In [7]:
model.eval()
with torch.no_grad():
    preds = model(X).argmax(1)
accuracy = (preds == y).float().mean()
print("Final train accuracy:", accuracy.item())


Final train accuracy: 0.9968652129173279


In [8]:
wrong = (preds != y).nonzero(as_tuple=True)[0]  # indices of wrong predictions
print("Number wrong:", len(wrong))
print("Indices of wrong predictions:", wrong.tolist())

# if you want to see true vs predicted labels:
for i in wrong.tolist():
    print(f"Index {i}: true={y[i].item()}, pred={preds[i].item()}")
names_true = encoder.inverse_transform(y[wrong].cpu())
names_pred = encoder.inverse_transform(preds[wrong].cpu())
for t, p in zip(names_true, names_pred):
    print(f"True: {t}, Predicted: {p}")


Number wrong: 4
Indices of wrong predictions: [22, 101, 211, 218]
Index 22: true=1, pred=2
Index 101: true=4, pred=5
Index 211: true=10, pred=6
Index 218: true=10, pred=6
True: Albin, Predicted: Benadik
True: Dio, Predicted: Edo
True: Milos, Predicted: Eliska
True: Milos, Predicted: Eliska


In [9]:
from proportional_split_xy import proportional_split_xy

## CrossEntropy Loss


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# === Load data ===
X = torch.load(embeddings).float()
y = torch.from_numpy(label_ids).long()   # must correspond to embeddings order

print(f"Loaded: X={X.shape}, y={y.shape}")

# === Train/Val split ===
X_train, X_val, y_train, y_val = proportional_split_xy(X, y, query_ratio=0.2)
train_ds = TensorDataset(X_train, y_train)
val_ds   = TensorDataset(X_val, y_val)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=16, shuffle=False)

# === Compute class weights to handle imbalance ===
classes = torch.unique(y).numpy()
weights = compute_class_weight('balanced', classes=classes, y=y.numpy())
class_weights = torch.tensor(weights, dtype=torch.float)
print("Class weights:", class_weights)

# === Define model ===
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=len(classes), hidden_dim=512, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = Classifier()
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# === Training loop ===
best_val_acc = 0
patience, patience_counter = 20, 0

for epoch in range(100):
    if epoch != 0:
        model.train()
    total_loss, correct = 0, 0
    for xb, yb in train_dl:
        if epoch != 0:
            optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        if epoch != 0:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * len(xb)
        correct += (out.argmax(1) == yb).sum().item()

    train_acc = correct / len(train_ds)

    # --- Validation ---
    model.eval()
    val_correct, val_loss = 0, 0
    with torch.no_grad():
        for xb, yb in val_dl:
            out = model(xb)
            loss = criterion(out, yb)
            val_loss += loss.item() * len(xb)
            val_correct += (out.argmax(1) == yb).sum().item()

    val_acc = val_correct / len(val_ds)
    print(f"Epoch {epoch:03d}: train_loss={total_loss/len(train_ds):.4f}, "
          f"train_acc={train_acc:.3f}, val_acc={val_acc:.3f}")

    # --- Early stopping ---
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

# === Evaluation on best model ===
model.load_state_dict(torch.load("best_model.pt"))
model.eval()
with torch.no_grad():
    preds = model(X_val).argmax(1)
val_acc = (preds == y_val).float().mean().item()
print(f"\n✅ Final validation accuracy: {val_acc:.3f}")

# === Optionally print misclassified examples ===
# wrong = (preds != y_val).nonzero(as_tuple=True)[0]
# print(f"Misclassified samples: {len(wrong)}/{len(y_val)}")
# for i in wrong[:10]:  # show up to 10
#     print(f"Index {i.item()}: true={y_val[i].item()}, pred={preds[i].item()}")

Loaded: X=torch.Size([260, 768]), y=torch.Size([260])
Class weights: tensor([3.7143, 0.6190, 0.3258, 2.6531, 2.3214, 0.7738, 2.3214, 0.5462, 1.4286,
        2.3214, 0.5628, 0.8442, 6.1905, 2.3214])
Epoch 001: train_loss=2.6488, train_acc=0.059, val_acc=0.018
Epoch 002: train_loss=2.6229, train_acc=0.123, val_acc=0.211
Epoch 003: train_loss=1.9763, train_acc=0.399, val_acc=0.281
Epoch 004: train_loss=1.4308, train_acc=0.468, val_acc=0.351
Epoch 005: train_loss=1.0689, train_acc=0.567, val_acc=0.193
Epoch 006: train_loss=0.6647, train_acc=0.700, val_acc=0.281
Epoch 007: train_loss=0.4661, train_acc=0.764, val_acc=0.281
Epoch 008: train_loss=0.3530, train_acc=0.877, val_acc=0.316
Epoch 009: train_loss=0.3162, train_acc=0.872, val_acc=0.368
Epoch 010: train_loss=0.2987, train_acc=0.901, val_acc=0.333
Epoch 011: train_loss=0.1674, train_acc=0.926, val_acc=0.368
Epoch 012: train_loss=0.1134, train_acc=0.961, val_acc=0.316
Epoch 013: train_loss=0.0923, train_acc=0.975, val_acc=0.404
Epoch 014